In [5]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests 
import random

import warnings
warnings.filterwarnings('ignore')

In [6]:
load_dotenv()

False

In [ ]:
llm = ChatOpenAI()

In [8]:
search_tool = DuckDuckGoSearchRun(region='us-en')

In [10]:
@tool
def calculator(first_num : float, second_num : float, operation : str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported Operations: add, sub, mul, div """

    try:
        if operation not in ['add', 'sub', 'mul', 'div']:
            return {'error': 'unsupported operation'}

        if operation == "add":
            result = first_num + second_num
        elif operation == 'sub':
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        else:
            if second_num == 0:
                return {'error': 'division by zero is not allowed'}
            result = first_num / second_num

        return {'result' : result}

    except Exception as e:
        return {'error' : str(e)}

In [11]:
@tool
def get_stock_price(symbol : str) -> dict:
    """fetch latest stock price for a given symbol e.g AAPL, TSLA
    using alpha vantage with the api key in the url """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    r = requests.get(url)
    return r.json()

In [ ]:
tools = [get_stock_price, search_tool, calculator]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call"""
    messages =  state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages" : [response]}

tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

In [ ]:
graph.add_edge(START, "chat_node")

graph.add_conditional_edge("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

In [ ]:
chatbot = graph.compile()

chatbot

In [ ]:


# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})

print(out["messages"][-1].content)

Hello! How can I assist you today?

# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is 2*3?")]})
print(out["messages"][-1].content)

The result of 2 multiplied by 3 is 6.

# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is the stock price of apple")]})
print(out["messages"][-1].content)

The stock price of Apple (AAPL) is $227.76.

# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="First find out the stock price of Apple using get stock price tool then use the calculator tool to find out how much will it take to purchase 50 shares?")]})
print(out["messages"][-1].content)

